In [1]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
import re

In [2]:
pd.set_option('display.max_colwidth', None)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"using compute device: {device}")

using compute device: cpu


In [3]:
#loading and converting dataset into dataframe
corpus_data = load_dataset('allenai/scifact', 'corpus')
claims_data = load_dataset('allenai/scifact', 'claims')
corpus_df = corpus_data['train'].to_pandas()
claims_df = claims_data['train'].to_pandas()

In [4]:
#normalizing IDs and text and converting abstract into a paragraph
corpus_df['doc_id'] = corpus_df['doc_id'].astype(str).str.strip()
claims_df['evidence_doc_id'] = claims_df['evidence_doc_id'].astype(str).str.strip()

def normalize_text(text):
    if pd.isna(text) or text is None:
        return ''
    text = re.sub(r'[\r\n\t]+', ' ',str(text))
    return re.sub(r'\s+', ' ', text).strip()

def make_paragraph(abstract_list):
    if isinstance(abstract_list, (list, np.ndarray)):
        return ' '.join(str(s).strip() for s in abstract_list)
    return str(abstract_list).strip()

corpus_df['clean_title'] = corpus_df['title'].apply(normalize_text)
corpus_df['clean_abstract'] = corpus_df['abstract'].apply(make_paragraph).apply(normalize_text)
claims_df['clean_claim'] = claims_df['claim'].apply(normalize_text)

In [5]:
# Concatenating title and abstract
corpus_df['full_text'] = corpus_df['clean_title'] + ". " + corpus_df['clean_abstract']

In [6]:
courpus_doc_ids = set(corpus_df['doc_id'])
eval_claims = claims_df[claims_df['evidence_doc_id'].isin(courpus_doc_ids) & (claims_df['evidence_doc_id'] != '')].copy() 

print(f"Corpus size: {len(corpus_df)} documents")
print(f"Evalution claims size: {len(eval_claims)} claims")

Corpus size: 5183 documents
Evalution claims size: 957 claims


In [7]:
import warnings
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

model_name = 'sentence-transformers/all-MiniLM-L6-v2'
model = SentenceTransformer(model_name, device=device)

print(f"Model loaded successfully: {model_name}")
print(f"Embedding dimension size: {model.get_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension size: 384


In [8]:
corpus_texts = corpus_df['full_text'].tolist()

print(f"Encoding {len(corpus_texts)} corpus documents on {device}...")
print('Please wait 1-2 minutes...')

corpus_embeddings = model.encode(
    corpus_texts,
    batch_size=64,
    show_progress_bar=False,
    convert_to_tensor=True,
    device=device
)

print("corpus encoding complete!")
print(f"corpus embeddings shape: {corpus_embeddings.shape}")

Encoding 5183 corpus documents on cpu...
Please wait 1-2 minutes...
corpus encoding complete!
corpus embeddings shape: torch.Size([5183, 384])


In [9]:
claims_texts =eval_claims['clean_claim'].tolist()

print(f"Encoding {len(claims_texts)} claims on {device}...")

claim_emdeddings = model.encode(
    claims_texts,
    batch_size=64,
    show_progress_bar=False,
    convert_to_tensor=True,
    device=device
)
print('Claim encoding complete!')
print(f"Claim embeddings shape: {claim_emdeddings.shape}")

Encoding 957 claims on cpu...
Claim encoding complete!
Claim embeddings shape: torch.Size([957, 384])


In [10]:
cosine_scores = util.cos_sim(claim_emdeddings, corpus_embeddings)

top_k = 10
top_k_results = torch.topk(cosine_scores, k=top_k, dim=1)

print(f"Similarity matrix shape: {cosine_scores.shape}")
print(f"Top-{top_k} indices shape: {top_k_results.indices.shape}")

Similarity matrix shape: torch.Size([957, 5183])
Top-10 indices shape: torch.Size([957, 10])


In [11]:
courpus_doc_ids_list = corpus_df['doc_id'].tolist()
true_target_doc_ids = eval_claims['evidence_doc_id'].tolist()

recall_at_1 = 0
recall_at_5 = 0
recall_at_10 = 0
mrr = 0.0

total_claims = len(eval_claims)

for claim_idx in range(total_claims):
    target_id = true_target_doc_ids[claim_idx]
    retrieved_indices = top_k_results.indices[claim_idx].tolist()
    retrieved_doc_ids = [courpus_doc_ids_list[idx] for idx in retrieved_indices]

    if target_id in retrieved_doc_ids[:1]:
        recall_at_1 += 1
    if target_id in retrieved_doc_ids[:5]:
        recall_at_5 += 1
    if target_id in retrieved_doc_ids[:10]:
        recall_at_10 += 10

    if target_id in retrieved_doc_ids:
        rank = retrieved_doc_ids.index(target_id) + 1
        mrr += 1.0/rank


recall_1_score = recall_at_1 / total_claims
recall_5_score = recall_at_5 / total_claims
recall_10_score = recall_at_10 / total_claims
mrr_score = mrr / total_claims


print("Dense Retrieval Baseline Performance")
print(f"Total Evaluated Claims: {total_claims}")
print(f"Recall@1  : {recall_1_score:.4f} ({recall_1_score * 100:.2f}%)")
print(f"Recall@5  : {recall_5_score:.4f} ({recall_5_score * 100:.2f}%)")
print(f"Recall@10 : {recall_10_score:.4f} ({recall_10_score * 100:.2f}%)")
print(f"MRR@10    : {mrr_score:.4f}")


Dense Retrieval Baseline Performance
Total Evaluated Claims: 957
Recall@1  : 0.6301 (63.01%)
Recall@5  : 0.8955 (89.55%)
Recall@10 : 9.4148 (941.48%)
MRR@10    : 0.7401


In [12]:
sample_idx = 0
sample_claim = eval_claims.iloc[sample_idx]
sample_target = sample_claim['evidence_doc_id']
sample_top_indices = top_k_results.indices[sample_idx].tolist()[:3]
sample_scores = top_k_results.values[sample_idx].tolist()[:3]

print("Claim ID:",sample_claim['id'])
print("claim:", sample_claim['clean_claim'])
print("Target Evidence Doc ID:", sample_target)
print('\n Top 3 Retrieved Paper: ')

for rank,(doc_idx, score) in enumerate(zip(sample_top_indices, sample_scores), start=1):
    retrieved_doc = corpus_df.iloc[doc_idx]
    is_hit = "Correct" if retrieved_doc['doc_id'] == sample_target else ""
    print(f'\nRank {rank} (score: {score:.4f}){is_hit}:')
    print('Doc ID:', retrieved_doc['doc_id'])
    print("Title:", retrieved_doc['clean_title'])

Claim ID: 2
claim: 1 in 5 million in UK have abnormal PrP positivity.
Target Evidence Doc ID: 13734012

 Top 3 Retrieved Paper: 

Rank 1 (score: 0.4328):
Doc ID: 3413083
Title: Patterns of chlamydia testing in different settings and implications for wider STI diagnosis and care: a probability sample survey of the British population

Rank 2 (score: 0.4238)Correct:
Doc ID: 13734012
Title: Prevalent abnormal prion protein in human appendixes after bovine spongiform encephalopathy epizootic: large scale survey

Rank 3 (score: 0.3704):
Doc ID: 21186109
Title: The missing cases of tuberculosis in Malawi: the contribution from cross-border registrations.


In [13]:
baseline_metrics_df = pd.DataFrame({
    'Model': [model_name],
    'Embedding_Dim': [model.get_embedding_dimension()],
    'Total_Claims': [total_claims],
    'Recall@1': [round(recall_1_score, 4)],
    'Recall@5': [round(recall_5_score, 4)],
    'Recall@10': [round(recall_10_score, 4)],
    'MRR@10': [round(mrr_score, 4)]
})

baseline_metrics_df

,Model,Embedding_Dim,Total_Claims,Recall@1,Recall@5,Recall@10,MRR@10
0,sentence-transformers/all-MiniLM-L6-v2,384,957,0.6301,0.8955,9.4148,0.7401


In [14]:
os.makedirs('results/tables', exist_ok=True)
baseline_metrics_df.to_csv('results/tables/baseline_dense_retrieval_metrics.csv', index=False)

print("Notebook 03 - Dense Retrieval Baseline complete")
print("Benchmark saved to: results/tables/baseline_dense_retrieval_metrics.csv")

Notebook 03 - Dense Retrieval Baseline complete
Benchmark saved to: results/tables/baseline_dense_retrieval_metrics.csv
